# 4. Analysis Pipeline
The core analysis module implements all four components of the analytical framework

## 4.1 CAV Computation

In [ ]:
import numpy as np
import torch
from sklearn.svm import LinearSVC

def compute_cav(acts_pos, acts_neg):
    """Compute Concept Activation Vector from positive/negative activations.

    Args:
        acts_pos: ndarray of shape (n_pos, n_features)
        acts_neg: ndarray of shape (n_neg, n_features)

    Returns:
        cav: Unit-length normalised SVM weight vector
    """
    X = np.vstack([acts_pos, acts_neg])
    y = np.concatenate([
        np.ones(len(acts_pos)),
        np.zeros(len(acts_neg))
    ])

    svm = LinearSVC(C=1.0, max_iter=10000, dual='auto')
    svm.fit(X, y)

    w = svm.coef_[0]
    cav = w / np.linalg.norm(w)
    return cav


## 4.2 TCAV Sensitivity Score

In [ ]:
def tcav_sensitivity(model, cav, class_idx, layer_idx,
                     dataloader, device='cpu'):
    """Compute TCAV sensitivity score for a concept-class pair.

    The score is the fraction of class-k inputs where the directional
    derivative along the CAV is positive.

    Returns:
        score: Float in [0, 1]
    """
    model.eval()
    cav_tensor = torch.tensor(cav, dtype=torch.float32,
                              device=device)

    positive_derivatives = 0
    total = 0

    for x, y in dataloader:
        mask = (y == class_idx)
        if mask.sum() == 0:
            continue

        x = x[mask].to(device)
        x.requires_grad = True

        # Forward pass to target layer
        out = x
        for i, lyr in enumerate(model.conv_layers):
            out = lyr(out)
            if i == layer_idx:
                break

        # Global average pooling to match CAV dimension
        h = out.mean(dim=(2, 3))

        # Continue to output
        for i in range(layer_idx + 1, len(model.conv_layers)):
            h = model.conv_layers[i](h)
        h = h.view(h.size(0), -1)
        logits = model.classifier(h)

        # Backprop for class logit
        class_logit = logits[:, class_idx].sum()
        model.zero_grad()
        class_logit.backward()

        # Directional derivative = gradient . CAV
        grad = out.grad.mean(dim=(2, 3)) if out.grad is not None else torch.zeros_like(h)
        directional = (grad * cav_tensor).sum(dim=1)
        positive_derivatives += (directional > 0).sum().item()
        total += len(x)

    return positive_derivatives / total if total > 0 else 0.0


## 4.3 Linear Probing

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

def linear_probe(acts, labels, n_splits=5):
    """Train linear probe with stratified k-fold cross-validation.

    Args:
        acts: ndarray of shape (n_samples, n_features)
        labels: ndarray of shape (n_samples,) with binary labels
        n_splits: Number of CV folds

    Returns:
        mean_accuracy: Mean test accuracy across folds
    """
    clf = LogisticRegression(
        penalty='l2',
        C=1.0,
        max_iter=1000,
        solver='lbfgs'
    )

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                          random_state=42)

    scores = cross_val_score(
        clf, acts, labels,
        cv=skf,
        scoring='accuracy',
        n_jobs=-1
    )

    return scores.mean(), scores.std()


## 4.4 RSA: RDM Computation

In [ ]:
def compute_rdm(acts):
    """Compute Representational Dissimilarity Matrix.

    Args:
        acts: ndarray of shape (n_stimuli, n_features)

    Returns:
        rdm: ndarray of shape (n_stimuli, n_stimuli)
               with 1 - Pearson correlation as dissimilarity
    """
    # Compute Pearson correlation matrix
    corr_matrix = np.corrcoef(acts)
    # Dissimilarity = 1 - correlation
    rdm = 1 - corr_matrix
    # Ensure diagonal is exactly 0
    np.fill_diagonal(rdm, 0)
    return rdm


def make_concept_rdm(labels):
    """Construct concept-model RDM from binary labels.

    Returns:
        rdm: 0 for same-concept pairs, 1 for different
    """
    n = len(labels)
    rdm = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if labels[i] != labels[j]:
                rdm[i, j] = 1.0
    return rdm


## 4.5 RSA: Kendall's Tau Correlation

In [ ]:
from scipy.stats import kendalltau

def rsa_correlation(rdm_net, rdm_concept):
    """Compute Kendall's tau-b between lower triangles of two RDMs.

    Args:
        rdm_net: Network Representational Dissimilarity Matrix
        rdm_concept: Concept-model RDM

    Returns:
        tau: Kendall's rank correlation coefficient
        pvalue: Two-sided p-value
    """
    # Extract lower triangle indices (excluding diagonal)
    idx = np.tril_indices_from(rdm_net, k=-1)

    # Flatten corresponding entries
    net_flat = rdm_net[idx]
    concept_flat = rdm_concept[idx]

    tau, pvalue = kendalltau(net_flat, concept_flat)
    return tau, pvalue


Bootstrap Confidence Intervals

In [ ]:
def rsa_bootstrap_ci(rdm_net, rdm_concept, n_bootstrap=1000,
                     confidence=0.95):
    """Bootstrap confidence interval for RSA correlation.

    Returns:
        (tau_observed, ci_lower, ci_upper)
    """
    n = rdm_net.shape[0]
    taus = []

    rng = np.random.RandomState(42)
    for _ in range(n_bootstrap):
        # Resample stimuli with replacement
        indices = rng.randint(0, n, size=n)

        # Subsample RDMs
        boot_net = rdm_net[np.ix_(indices, indices)]
        boot_concept = rdm_concept[np.ix_(indices, indices)]

        tau, _ = rsa_correlation(boot_net, boot_concept)
        taus.append(tau)

    alpha = 1 - confidence
    ci_lower = np.percentile(taus, 100 * alpha / 2)
    ci_upper = np.percentile(taus, 100 * (1 - alpha / 2))

    tau_obs, _ = rsa_correlation(rdm_net, rdm_concept)
    return tau_obs, ci_lower, ci_upper


## 4.6 Targeted Ablation

In [ ]:
def targeted_ablation(model, images, labels, concept_mask,
                     layer_idx, top_pct=0.10, device='cpu'):
    """Ablate top concept-aligned units and measure accuracy change.

    Args:
        model: Trained MNIST_CNN
        images: Input batch tensor
        labels: True labels
        concept_mask: Boolean mask for concept-positive examples
        layer_idx: Target convolutional layer
        top_pct: Fraction of units to ablate

    Returns:
        (acc_before, acc_after, delta_acc)
    """
    model.eval()

    # Baseline accuracy (no ablation)
    with torch.no_grad():
        logits_orig = model(images.to(device))
        preds_orig = logits_orig.argmax(dim=1)
        acc_before = (preds_orig.cpu() == labels).float().mean().item()

    # Identify top concept-correlated units
    with torch.no_grad():
        acts = []
        def hook_fn(module, input, output):
            acts.append(output)

        handle = model.conv_layers[layer_idx].register_forward_hook(hook_fn)
        _ = model(images.to(device))
        handle.remove()

        activation = acts[0].mean(dim=(2, 3))  # (B, C)

        # Correlation with concept labels
        concept_labels = concept_mask.float().to(device)
        correlations = []
        for c in range(activation.shape[1]):
            corr = torch.corrcoef(
                torch.stack([activation[:, c], concept_labels])
            )[0, 1].abs()
            correlations.append(corr.item())

        # Select top units
        n_units = len(correlations)
        n_ablate = max(1, int(top_pct * n_units))
        top_indices = np.argsort(correlations)[-n_ablate:]

    # Ablation hook: zero out selected channels
    def ablation_hook(module, input, output):
        output[:, top_indices, :, :] = 0
        return output

    handle = model.conv_layers[layer_idx].register_forward_hook(ablation_hook)

    with torch.no_grad():
        logits_abl = model(images.to(device))
        preds_abl = logits_abl.argmax(dim=1)
        acc_after = (preds_abl.cpu() == labels).float().mean().item()

    handle.remove()

    return acc_before, acc_after, acc_after - acc_before


## 4.6. Counterfactual Injection

In [ ]:
def counterfactual_injection(model, images, labels, cav,
                            layer_idx, alpha=1.0, device='cpu'):
    """Inject concept direction into activations and measure logit change.

    Args:
        model: Trained MNIST_CNN
        images: Input batch tensor
        labels: True labels
        cav: Concept Activation Vector (unit-length)
        layer_idx: Target layer for injection
        alpha: Injection strength multiplier

    Returns:
        mean_logit_change: Average change in target class logits
    """
    model.eval()
    cav_t = torch.tensor(cav, dtype=torch.float32, device=device)

    # Capture original logits
    with torch.no_grad():
        logits_orig = model(images.to(device))

    # Injection hook
    def injection_hook(module, input, output):
        # GAP to CAV dimension, inject, expand back
        b, c, h, w = output.shape
        gap = output.mean(dim=(2, 3))  # (B, C)
        gap_modified = gap + alpha * cav_t.unsqueeze(0)
        # Expand back to spatial: broadcast across H, W
        diff = (gap_modified - gap).view(b, c, 1, 1)
        output = output + diff.expand(b, c, h, w)
        return output

    handle = model.conv_layers[layer_idx].register_forward_hook(
        injection_hook)

    with torch.no_grad():
        logits_mod = model(images.to(device))

    handle.remove()

    # Measure change in ground-truth class logits
    target_logits_orig = logits_orig.gather(1, labels.view(-1, 1).to(device))
    target_logits_mod = logits_mod.gather(1, labels.view(-1, 1).to(device))

    logit_change = (target_logits_mod - target_logits_orig).squeeze()

    return logit_change.mean().item(), logit_change.std().item()
